### RAG 심화
- 중복 문서 문제 : 비슷한 내용의 chunk 여러개 반환되어 컨텍스트 낭비
- 검색 : 시멘틱 유사도만으로는 정확한 키워드 매칭 어려움
- 구조적 질의 불가 : ex) 2024년 이후 계약 금액이 1억 이상인 제품 찾기 = 메타필터 처리 불가
- 노이즈 청크 : 관련성이 낮은 청크가 LLM에게 전달되어 환각 유발

In [1]:
# 라이브러리 로드
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda,
    RunnableParallel,
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import (
    InMemoryChatMessageHistory,
    BaseChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os
import gradio as gr

# 모델(LLM, Embedding)
from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    WebBaseLoader,
    DirectoryLoader,
)
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import (
    EnsembleRetriever, 
    ContextualCompressionRetriever, 
    BM25Retriever
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


c:\souce\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\soldesk\AppData\Local\Temp\ipykernel_22480\1330226653.py:32: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")

In [4]:
from langchain_openai import ChatOpenAI

# HuggingFace model

hugging_llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    api_key=hf_token,
    base_url="https://router.huggingface.co/v1",
    temperature= 0
)
# 유료 LLM 선언

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

# 로컬 LLM 선언
qwen_llm = ChatOllama(model="qwen3.5:4b",temperature= 0)

exaone_llm = ChatOllama(model="exaone3.5:2.4b",temperature= 0)

In [5]:
ollama_enbedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
    )

In [6]:
# pdf => chunks 반환 함수
def create_chunk_from_pdf(pdf_path,chunk_size=500,chunk_overlap=50):
    loder = PyPDFLoader(pdf_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(loder.load())

    # 공백 제거
    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
    return chunks

def create_vectorstore(chunks, embeddings, collection_name,persist_directory='./db/chroma_db'):
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )

def create_retriever(vectorstore,search_type="similarity",k=3,fetch_k=20,lambda_mult=0.5):
    kwargs = {"k":k}

    if search_type=="mmr":
        kwargs['fetch_k']= fetch_k
        kwargs['lambda_mult'] = lambda_mult

    return vectorstore.as_retriever(search_type=search_type,search_kwargs=kwargs)

def print_retrieved_docs(title, retriever, query):
    docs = retriever.invoke(query)
    
    print("\n"+"="*50)
    print(title)
    print("="*50)

    for i, doc in enumerate(docs):
        print(f"\n[chunk {i}]")
        print(doc.page_content)
        print(f"\nPage : {doc.metadata.get("page")}")

### 1. 임베딩 코델, 청크 사이즈, 오버랩

In [ ]:
chunk1 = create_chunk_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf",1000,100)
chunk2 = create_chunk_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf",300,30)

print(f"분할된 청크 수 : {len(chunk1)}")
print(f"분할된 청크 수 : {len(chunk2)}")

분할된 청크 수 :118
분할된 청크 수 :378


In [20]:
vectorstore1 = create_vectorstore(chunk1,watsonx_enbedding,collection_name="gtp_research_watson1")
vectorstore2 = create_vectorstore(chunk2,watsonx_enbedding,collection_name="gtp_research_watson2")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where ci use chatGPT?'

print_retrieved_docs('chunk=1000,overlap=100',watson1_retriever, query)
print_retrieved_docs('chunk=300,overlap=30',watson2_retriever, query)


chunk=1000,overlap=100

[chunk 0]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 1]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 2]
development.
2 Related work of ChatGPT
In thi

In [21]:
vectorstore1 = create_vectorstore(chunk1,watsonx_enbedding,collection_name="gtp_research_watson1")
vectorstore2 = create_vectorstore(chunk2,ollama_enbedding,collection_name="gtp_research_watson2")

watson1_retriever = create_retriever(vectorstore1)
watson2_retriever = create_retriever(vectorstore2)

query = 'where ci use chatGPT?'

print_retrieved_docs('watsonx_enbedding',watson1_retriever, query)
print_retrieved_docs('ollama_enbedding',watson2_retriever, query)


watsonx_enbedding

[chunk 0]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 1]
development.
2 Related work of ChatGPT
In this section, we review the latest research related to the application, ethics,
and evaluation of ChatGPT.
2.1 Application of ChatGPT
2.1.1 Question And Answering
In the education ﬁeld
ChatGPT is commonly used for question and answers testing in the edu-
cation sector. Users can use ChatGPT to learn, compare and verify answers
for diﬀerent academic subjects such as physics, mathematics, and chemistry,
4

[chunk 2]
development.
2 Related work of ChatGPT
In this sec

### 2. MMR(Maximal Marginal Relevace) Retriever
- 관련성(Relevace)과 다양성(Diversity) 고려
- 법률 문서, 기술 메뉴얼 처럼 유사 내용이 반복되는 문서에 효과적임
- 동작과정
    - research => embedding
    - vectorstore에서 research과 유사한 상위 fetch_k(후보 문서)를 추출
    - fetch_k 에서 MMR 점수 계산 => 가장 높은 문서 추출
    - 남은 fetch_에서 MMR 점수 계산 => 높은 문서 추출
    - 추출한 높은 문서에서 최종 k 반환

In [ ]:


chunk1 = create_chunk_from_pdf("./data/2026 상 삼성전자 DX부문 직무기술서.pdf",500,50)

vectorstore1 = create_vectorstore(chunk1,watsonx_enbedding,collection_name="samsung_watson1",persist_directory="./db/watson_chroma")

mmr_retriever = create_retriever(vectorstore1,search_type="mmr",k=5)
similarity_retriever = create_retriever(vectorstore1,k=5)

query = '마케팅 - 제품/서비스 마케팅 포지션은?'

print_retrieved_docs('MMR',mmr_retriever, query)
print_retrieved_docs('Similarity',similarity_retriever, query)


MMR

[chunk 0]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과적인
커뮤니케이션 방법으로 전달하여 목표한 경영성과를 창출하고 브랜드 가치를 제고합니다

Page : 14

[chunk 1]
국내영업마케팅
국내의 각 분야별 영업 채널을 발굴
 지원하여 성과 창출과 지속 성장을 추구하는 동시에
 한국 시장에
대한 심도있는 분석을 통해 삼성전자
 부문 제품의 마케팅 전략을 수립 ⋅ 적용하고 글로벌
시장으로의 확산 기반을 마련합니다

Page : 24

[chunk 2]
해외영업
고객과 시장
 제품에 대한 이해를 바탕으로 시장 수요와 경쟁환경을 분석하여 국가
 거래선별 목표 설정
영업전략 수립
 신규 제품
 영업 채널을 발굴하고 판매전략 수립 및 실행을 통해 매출 극대화에
기여합니다

Page : 18

[chunk 3]
구매
제품 생산에 필요한 자원
 부품
 설비 및 제품
 을 최적의 품질과 가격으로
협상
 구매하고 시장의 수요 및 생산 계획에 맞춰 적기 공급하여 회사 경영에 기여합니다

Page : 26

[chunk 4]
품질/서비스
신제품 개발 신뢰성 검증
 공정 불량 검출
 고객 서비스 지원
 부품 협력사 관리 등 불량 없는 제품 생산
및 고객만족 실현을 위한 솔루션을 수립하여 제공합니다

Page : 12

Similarity

[chunk 0]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과적인
커뮤니케이션 방법으로 전달하여 목표한 경영성과를 창출하고 브랜드 가치를 제고합니다

Page : 14

[chunk 1]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과적인
커뮤니케이션 방법으로 전달하여 목표한 경영성과를 창출하고 브랜드 가치를 제고합니다

Page : 14

[chunk 2]
마케팅
마켓 센싱 및 정보 분석 결과를 바탕으로
 당사 제품과 서비스의 차별화 가치를 소비자에게 효과

### 3. SelfQuery Retriever
- 자연어 질문을 분석하여 시멘틱 검색 쿼리와 메타데이터필터를 LLM이자동으로 생성하게하는 고급 Retriever
- 질문 : 2023년 이후 계약금액이 1억 이상인 계약 찾아줘 => LLM
    - 시멘틱 검섹 쿼리 :계약
    - filtter:{year > = 2023, 계약금액 >= 100000 ~}

In [35]:
!pip3 install lark


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata={
            "year" :2025,
            "department":"marketing"
        }
    ),
    Document(
        page_content="AI 연구 개발 직무입니다.",
        metadata={
            "year" :2024,
            "department":"ai"
        }
    ),
    Document(
        page_content="백엔드 개발 직무입니다.",
        metadata={
            "year" :2025,
            "department":"developer"
        }
    ),
]


metadata_feild_info = [
    AttributeInfo(name="year",description="문서 작성 연도", type="integer"),
    AttributeInfo(name="department",description="wlran qntj", type="string"),
]

document_content_description = "회사 내부 문서 및 직무 자료"

In [9]:
vectorstore1 = create_vectorstore(docs,watsonx_enbedding,collection_name="selfquery",persist_directory="./db/watson_chroma")

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    document_contents=document_content_description,
    metadata_field_info=metadata_feild_info,
    verborse = True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator()
)

In [13]:
question = "2022년 이후 ai 부서 직무 찾아줘"

self_query_retriever.invoke(question)

[Document(id='b1d5e860-6596-4f49-9e25-dc8a398f77a2', metadata={'year': 2024, 'department': 'ai'}, page_content='AI 연구 개발 직무입니다.'),
 Document(id='a88f9449-8d27-4983-9435-d4c549a9598b', metadata={'year': 2024, 'department': 'ai'}, page_content='AI 연구 개발 직무입니다.'),
 Document(id='369e1452-d746-47be-bfcd-11cd7786ea0c', metadata={'year': 2024, 'department': 'ai'}, page_content='AI 연구 개발 직무입니다.')]